# Linear Regression — Mathematics

**Goal.** Build the linear-regression / OLS theory from the ground up: write the model in matrix form, derive the MSE loss from probabilistic assumptions, compute its gradient and Hessian, prove the loss is convex, state existence and uniqueness, derive the closed-form solution, interpret it geometrically via the hat matrix, and handle the singular case via the pseudoinverse.

**Role of this notebook.** Pure mathematics — symbols, definitions, derivations, theorems. The intuition belongs in `01_intuition.ipynb` (pictures, sliders, residuals). The implementation belongs in `05_hands_on_programming.ipynb`. This notebook stays text-only on purpose.

**Prerequisites.** `01_intuition.ipynb` — the visual picture of data, lines, residuals, and the bowl-shaped loss. Comfort with multivariable calculus and basic linear algebra (matrix transpose, symmetric matrices, Gram matrices, eigenvalues, orthogonal projection, the SVD).

**Stage map.** `01_intuition` → **`02_mathematics`** → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

**Theorem stack.**

1. **Model.** How do we write the linear model in matrix form?
2. **Loss.** Why MSE? (Negative log-likelihood under Gaussian noise.)
3. **Gradient and Hessian.** Where does ∇L point, and why is L convex?
4. **Optimum.** What characterises the minimum (normal equations), and when is it unique?
5. **Geometric meaning.** What is θ\* as a linear-algebra object? (Projection of y onto Col(X) via the hat matrix.)
6. **Singular case.** What replaces (XᵀX)⁻¹ when XᵀX is singular? (Moore–Penrose pseudoinverse via SVD.)

---

**Reading conventions.** Single-line equations that name an object appear in a blockquote so they stand out from prose. Multi-line derivations with aligned `=` appear in code blocks. Theorem statements appear in blockquotes; proofs are signed off with ∎. Equations are numbered, e.g. (3.2), only when later sections refer back to them.

## 0. Notation

Every symbol used later, defined once.

| Symbol | Type | Meaning |
|---|---|---|
| n | scalar ∈ ℕ | number of training examples |
| p | scalar ∈ ℕ | number of features (including the intercept after the bias trick) |
| xᵢ | vector ∈ ℝᵖ | feature vector of the i-th example |
| yᵢ | scalar ∈ ℝ | target of the i-th example |
| X | matrix ∈ ℝⁿˣᵖ | design matrix; row i is xᵢᵀ |
| y | vector ∈ ℝⁿ | target vector with entries yᵢ |
| θ | vector ∈ ℝᵖ | model parameters (weights) |
| ŷᵢ | scalar | model prediction for example i; ŷᵢ = xᵢᵀ θ |
| ŷ | vector ∈ ℝⁿ | prediction vector; ŷ = X θ |
| rᵢ | scalar | residual for example i; rᵢ = ŷᵢ − yᵢ |
| r\* | vector ∈ ℝⁿ | residual vector at the optimum; r\* = ŷ\* − y |
| L(θ) | scalar function | the MSE loss; L : ℝᵖ → ℝ⁺ |
| ∇L(θ) | vector ∈ ℝᵖ | gradient of L at θ |
| ∇²L(θ) | matrix ∈ ℝᵖˣᵖ | Hessian of L at θ |
| θ\* | vector ∈ ℝᵖ | a minimiser of L |
| H | matrix ∈ ℝⁿˣⁿ | hat matrix; H := X (XᵀX)⁻¹ Xᵀ |
| X⁺ | matrix ∈ ℝᵖˣⁿ | Moore–Penrose pseudoinverse of X |
| εᵢ | scalar (random) | noise term in the probabilistic model |
| σ² | scalar > 0 | noise variance |
| ‖·‖ | scalar | Euclidean (ℓ²) norm |
| ⟨·, ·⟩ | scalar | Euclidean inner product; ⟨a, b⟩ = aᵀ b |
| Col(X) | subspace ⊆ ℝⁿ | column space of X (span of its columns) |
| Null(X) | subspace ⊆ ℝᵖ | null space of X; { v : X v = 0 } |
| rank(X) | scalar ∈ ℕ | dim Col(X); for X ∈ ℝⁿˣᵖ, rank(X) ≤ min(n, p) |

**Conventions used throughout.**

- All vectors are *column* vectors; lowercase letters denote vectors, uppercase letters matrices.
- Aᵀ is the transpose of A; A⁻¹ its inverse (when it exists); A⁺ its Moore–Penrose pseudoinverse.
- Subscripts index examples (xᵢ); commas separate row/column indices when both are needed (Xᵢⱼ).
- A matrix is **symmetric** if Aᵀ = A; **positive semi-definite (PSD)** if vᵀ A v ≥ 0 for all v; **positive definite (PD)** if vᵀ A v > 0 for all v ≠ 0.
- An n × n matrix P is an **orthogonal projection** iff Pᵀ = P and P² = P.
- Multiplication: a centred dot `·` separates *scalar* factors (e.g. (1/n) · L); juxtaposition is reserved for matrix–vector and matrix–matrix products (e.g. X θ, XᵀX).

## 1. Model: linear regression in matrix form

### 1.1 Definition (linear model)

A **linear model** assumes the prediction is a linear combination of features. For each example i ∈ {1, …, n} and an unknown parameter vector θ ∈ ℝᵖ:

> ŷᵢ = xᵢ₁ θ₁ + xᵢ₂ θ₂ + ⋯ + xᵢₚ θₚ = ⟨xᵢ, θ⟩ = xᵢᵀ θ.

### 1.2 Matrix form

Stack the per-example equations row by row. Define the **design matrix** X ∈ ℝⁿˣᵖ as the matrix whose i-th row equals xᵢᵀ, and the **target vector** y ∈ ℝⁿ as the column vector with entries yᵢ. Then the n scalar equations ŷᵢ = xᵢᵀ θ collapse into a single matrix–vector product:

> ŷ = X θ ∈ ℝⁿ.   (1.1)

### 1.3 Bias trick

A real model has an intercept (bias) term θ₀, so that

> ŷᵢ = θ₀ + xᵢ₁ θ₁ + ⋯ + xᵢₚ θₚ.

Absorb θ₀ into θ by prepending a column of 1s to the feature matrix. Let 𝟙 := (1, 1, …, 1)ᵀ ∈ ℝⁿ. Then

> X̃ := [ 𝟙 | X ] ∈ ℝⁿˣ⁽ᵖ⁺¹⁾,    θ̃ := (θ₀, θ₁, …, θₚ)ᵀ ∈ ℝᵖ⁺¹,    ŷ = X̃ θ̃.

From here on we drop the tildes and assume X already includes the bias column; p denotes the total number of weights (including θ₀).

## 2. Loss: mean squared error

### 2.1 Definition (residual and MSE)

The **residual** for example i is the signed error

> rᵢ(θ) := ŷᵢ − yᵢ = xᵢᵀ θ − yᵢ.

The **mean squared error** is the averaged squared residual. Written equivalently in scalar, vector, and quadratic-expansion forms:

```
L(θ) = (1/n) · ∑ᵢ rᵢ(θ)²                                       (scalar form)
     = (1/n) · ‖X θ − y‖²                                       (vector form)         (2.1)
     = (1/n) · (θᵀ XᵀX θ  −  2 yᵀ X θ  +  yᵀ y)                 (expanded quadratic)  (2.2)
```

**Ordinary least squares (OLS)** is the optimisation problem

> θ\* ∈ argmin_{θ ∈ ℝᵖ} L(θ).

### 2.2 Theorem (MSE is the Gaussian negative log-likelihood)

Squaring is not an aesthetic choice — it is forced by a probabilistic model.

> **Theorem 2.2.** Assume the data-generating process
>
> yᵢ = xᵢᵀ θ + εᵢ,    εᵢ ~ 𝒩(0, σ²)  i.i.d.,    i = 1, …, n.
>
> Then the maximum-likelihood estimator of θ given (X, y) is identical to the OLS minimiser of L(θ).

**Proof.** Conditional on X, the residuals εᵢ = yᵢ − xᵢᵀ θ are i.i.d. 𝒩(0, σ²), so the joint density factorises:

```
p(y | X, θ) = ∏ᵢ (2π σ²)^(−1/2) · exp( −(yᵢ − xᵢᵀ θ)² / (2σ²) ).
```

Take logarithms and use the definition of L(θ):

```
ℓ(θ) := log p(y | X, θ)
      = −(n/2) · log(2π σ²)  −  (1 / 2σ²) · ∑ᵢ (yᵢ − xᵢᵀ θ)²
      = −(n/2) · log(2π σ²)  −  (n / 2σ²) · L(θ).
```

The first term does not depend on θ, and (n / 2σ²) > 0 is a positive constant. Therefore

> argmax_θ ℓ(θ)  =  argmin_θ L(θ),

which is exactly the OLS minimiser. ∎

**Remarks.**

- Different noise distributions give different losses: εᵢ ~ Laplace(0, b) yields mean absolute error (MAE); a Gaussian–Laplacian mixture yields the Huber loss.
- Differentiability of L is automatic (each square is C^∞), and a single residual of magnitude 10 contributes the same to L as 100 residuals of magnitude 1 — a quadratic penalty for outliers, baked in for free.

## 3. Gradient and Hessian

### 3.1 Matrix calculus identities

Two identities power all of OLS calculus. Both are stated for column vectors θ ∈ ℝᵖ.

**Identity (i) — linear form.** For a constant vector b ∈ ℝᵖ,

> ∇_θ (bᵀ θ) = b.

*Proof.* bᵀ θ = ∑ⱼ bⱼ θⱼ, hence ∂(bᵀ θ)/∂θₖ = bₖ for each k. Stacking gives the column vector b. ∎

**Identity (ii) — quadratic form.** For a constant matrix A ∈ ℝᵖˣᵖ,

> ∇_θ (θᵀ A θ) = (A + Aᵀ) θ.

*Proof.* Write θᵀ A θ = ∑ⱼₖ Aⱼₖ θⱼ θₖ. Differentiate with respect to θₗ:

```
∂/∂θₗ ( ∑ⱼₖ Aⱼₖ θⱼ θₖ )  =  ∑ₖ Aₗₖ θₖ  +  ∑ⱼ Aⱼₗ θⱼ  =  (A θ)ₗ + (Aᵀ θ)ₗ.
```

Stacking over l gives (A + Aᵀ) θ. ∎

*Special case.* When A is symmetric (A = Aᵀ), the identity reduces to ∇_θ (θᵀ A θ) = 2 A θ.

### 3.2 Theorem (gradient of L)

> **Theorem 3.2.**   ∇L(θ) = (2/n) · Xᵀ (X θ − y).   (3.1)

**Proof.** Start from the expanded form (2.2):

```
L(θ) = (1/n) · ( θᵀ XᵀX θ  −  2 yᵀ X θ  +  yᵀ y ).
```

Differentiate term by term.

- *Quadratic term.* Apply Identity (ii) with A = XᵀX. The matrix XᵀX is symmetric ((XᵀX)ᵀ = XᵀX), so its gradient is 2 XᵀX θ.
- *Linear term.* Rewrite −2 yᵀ X θ = −2 (Xᵀ y)ᵀ θ and apply Identity (i) with b = Xᵀ y, yielding −2 Xᵀ y.
- *Constant term.* yᵀ y does not depend on θ, so its gradient is 0.

Combining:

```
∇L(θ)  =  (1/n) · ( 2 XᵀX θ  −  2 Xᵀ y )  =  (2/n) · Xᵀ (X θ − y).   ∎
```

### 3.3 Theorem (Hessian of L)

> **Theorem 3.3.**   ∇²L(θ) = (2/n) · XᵀX,    independent of θ.

**Proof.** From (3.1), ∇L(θ) = (2/n) · (XᵀX θ − Xᵀ y). The map θ ↦ (2/n) · XᵀX θ is linear in θ with constant Jacobian (2/n) · XᵀX; the constant term −(2/n) · Xᵀ y has zero Jacobian. ∎

### 3.4 Theorem (convexity of L)

> **Theorem 3.4.**
> 
> 1. L is convex on ℝᵖ.
> 2. L is **strictly** convex if and only if rank(X) = p.

**Proof.** Standard fact (multivariable calculus): a twice-differentiable function on ℝᵖ is convex iff its Hessian is PSD everywhere, and strictly convex iff its Hessian is PD everywhere. By Theorem 3.3, ∇²L = (2/n) · XᵀX, so it suffices to study XᵀX.

*(1) XᵀX is PSD.* For any v ∈ ℝᵖ,

```
vᵀ (XᵀX) v  =  (X v)ᵀ (X v)  =  ‖X v‖²  ≥  0.
```

Hence XᵀX is PSD, ∇²L is PSD, and L is convex.

*(2) PD ⟺ full column rank.* From the same identity, XᵀX is PD iff ‖X v‖² > 0 for every v ≠ 0, iff X v = 0 implies v = 0, iff Null(X) = {0}, iff the columns of X are linearly independent, iff rank(X) = p. ∎

**Reading.** The "bowl" of `01_intuition.ipynb` §3 is now a theorem: L is a convex quadratic, *strictly* convex whenever the features are linearly independent. Strict convexity forces a unique minimiser; mere convexity only guarantees a convex *set* of minimisers.

## 4. Optimum: the normal equations

### 4.1 First-order optimality

Since L is convex (Theorem 3.4 part 1), every critical point of L is a *global* minimiser. So we set ∇L(θ) = 0 in (3.1) and cancel the constant (2/n):

> XᵀX θ = Xᵀ y.   (4.1)

Equation (4.1) is the system of **normal equations** for OLS.

### 4.2 Theorem (existence and uniqueness)

> **Theorem 4.2.** Let X ∈ ℝⁿˣᵖ, y ∈ ℝⁿ, and L(θ) = (1/n) · ‖X θ − y‖². Define Θ\* := argmin_θ L(θ). Then
> 
> 1. **(Existence.)** Θ\* is non-empty.
> 2. **(Uniqueness.)** |Θ\*| = 1   ⟺   rank(X) = p   ⟺   XᵀX is invertible.
> 3. **(Closed form.)** When rank(X) = p, the unique minimiser is
> 
>     θ\* = (XᵀX)⁻¹ Xᵀ y.   (4.2)
> 
> When rank(X) < p, Θ\* is an affine subspace of ℝᵖ of dimension p − rank(X). §6 selects its unique minimum-norm element via the pseudoinverse.

**Proof.**

*(1) Existence.* A vector θ minimises L iff it satisfies the normal equations (4.1). The system is consistent because Xᵀ y belongs to the column space of XᵀX: indeed Col(XᵀX) = Col(Xᵀ) (a consequence of Null(X) = Null(XᵀX) and rank–nullity), and Xᵀ y ∈ Col(Xᵀ) trivially. Hence Θ\* ≠ ∅.

*(2) Uniqueness.* By Theorem 3.4 part 2, L is strictly convex iff rank(X) = p. A strictly convex function has at most one minimiser; combined with (1), it has exactly one. Conversely, suppose rank(X) < p and pick θ\* ∈ Θ\* together with any v ∈ Null(X) \ {0}. Then X v = 0, so for every α ∈ ℝ,

```
L(θ\* + α v)  =  (1/n) · ‖X(θ\* + α v) − y‖²  =  (1/n) · ‖X θ\* − y‖²  =  L(θ\*).
```

Hence θ\* + α v ∈ Θ\* for every α, giving |Θ\*| = ∞.

*(3) Closed form.* Under rank(X) = p, XᵀX is invertible (Theorem 3.4), so (4.1) has the unique solution θ\* = (XᵀX)⁻¹ Xᵀ y. ∎

### 4.3 Computational note

Although (XᵀX)⁻¹ Xᵀ y is the cleanest *mathematical* expression for θ\*, it is rarely the best *numerical* path. Forming XᵀX squares the condition number of X, so direct solvers based on a QR or SVD factorisation of X (without forming XᵀX) are preferred in practice. The closed form also costs Θ(n p² + p³) time and Θ(p²) memory, which is impractical for large p — this is what motivates `03_optimization.ipynb`.

## 5. Geometric meaning: hat matrix and orthogonal projection

### 5.1 Residual orthogonality

Rewrite the normal equations (4.1) as

> Xᵀ (X θ\* − y) = 0,    i.e.    Xᵀ r\* = 0,    where r\* := X θ\* − y.   (5.1)

Equation (5.1) says r\* is orthogonal to every column of X, hence to every vector in Col(X). This is the linear-algebra signature of an **orthogonal projection**: ŷ\* = X θ\* is the unique vector in Col(X) such that y − ŷ\* ⊥ Col(X).

### 5.2 Definition (hat matrix)

Assume rank(X) = p, so (XᵀX)⁻¹ exists. Substituting θ\* = (XᵀX)⁻¹ Xᵀ y into ŷ\* = X θ\*:

> ŷ\* = X (XᵀX)⁻¹ Xᵀ y = H y,    where    H := X (XᵀX)⁻¹ Xᵀ ∈ ℝⁿˣⁿ.   (5.2)

H is called the **hat matrix** — it puts the hat on y.

### 5.3 Theorem (H is the orthogonal projection onto Col(X))

> **Theorem 5.3.** The hat matrix H satisfies
> 
> 1. **Symmetry.**   Hᵀ = H.
> 2. **Idempotence.**   H² = H.
> 3. **Range.**   Im(H) = Col(X).
> 4. **Spectrum.**   Every eigenvalue of H is 0 or 1, and rank(H) = trace(H) = p.
> 
> Properties (1)–(2) together characterise an orthogonal projection matrix in ℝⁿ.

**Proof.**

*(1) Symmetry.* Use (A B)ᵀ = Bᵀ Aᵀ and the symmetry of XᵀX (so ((XᵀX)⁻¹)ᵀ = (XᵀX)⁻¹):

```
Hᵀ  =  ( X (XᵀX)⁻¹ Xᵀ )ᵀ  =  X ( (XᵀX)⁻¹ )ᵀ Xᵀ  =  X (XᵀX)⁻¹ Xᵀ  =  H.
```

*(2) Idempotence.* Expand H², cancelling the central (XᵀX)(XᵀX)⁻¹ = Iₚ:

```
H²  =  X (XᵀX)⁻¹ Xᵀ X (XᵀX)⁻¹ Xᵀ  =  X (XᵀX)⁻¹ Xᵀ  =  H.
```

*(3) Range.* For any y, H y = X · [ (XᵀX)⁻¹ Xᵀ y ] ∈ Col(X), so Im(H) ⊆ Col(X). Conversely, take z ∈ Col(X); write z = X w for some w ∈ ℝᵖ. Then

```
H z  =  X (XᵀX)⁻¹ Xᵀ X w  =  X w  =  z,
```

so Col(X) ⊆ Im(H). Combining, Im(H) = Col(X).

*(4) Spectrum.* If H v = λ v with v ≠ 0, then H² v = λ² v; using H² = H,

```
λ² v  =  λ v   ⟹   λ (λ − 1) v = 0   ⟹   λ ∈ {0, 1}.
```

The multiplicity of the eigenvalue 1 equals dim Im(H) = dim Col(X) = rank(X) = p. The trace of H equals the sum of its eigenvalues = p, which also equals rank(H). ∎

### 5.4 Corollary (Pythagoras)

> **Corollary 5.4.**   ‖y‖² = ‖ŷ\*‖² + ‖r\*‖².   (5.3)

**Proof.** From (5.1), r\* ⊥ Col(X); from (5.2), ŷ\* ∈ Col(X). Therefore ⟨ŷ\*, r\*⟩ = 0. Now decompose y = ŷ\* − r\* (since r\* = ŷ\* − y):

```
‖y‖²  =  ‖ŷ\* − r\*‖²  =  ‖ŷ\*‖² − 2 ⟨ŷ\*, r\*⟩ + ‖r\*‖²  =  ‖ŷ\*‖² + ‖r\*‖².   ∎
```

**Reading.** OLS is not just "the line that minimises a sum". It is *the* orthogonal projection of y onto the subspace Col(X) of achievable predictions. Pythagoras decomposes the total squared length of y into an **explained** part ‖ŷ\*‖² and an **unexplained** part ‖r\*‖² — the seed of the R² coefficient developed in `04_statistics.ipynb`.

## 6. Singular case: SVD and the Moore–Penrose pseudoinverse

When rank(X) < p (the *multicollinear* regime), XᵀX is singular, the closed form (4.2) is undefined, and Θ\* is an infinite affine subspace by Theorem 4.2 part 2. The singular value decomposition lets us still write down a canonical minimiser.

### 6.1 The singular value decomposition

> **Theorem 6.1 (SVD).** Every X ∈ ℝⁿˣᵖ admits a factorisation
> 
> X = U Σ Vᵀ,
> 
> where U ∈ ℝⁿˣⁿ and V ∈ ℝᵖˣᵖ are orthogonal (Uᵀ U = Iₙ, Vᵀ V = Iₚ) and Σ ∈ ℝⁿˣᵖ is diagonal with non-negative entries
> 
> σ₁ ≥ σ₂ ≥ ⋯ ≥ σ_{min(n,p)} ≥ 0.
> 
> The non-zero σᵢ are the **singular values** of X; exactly r := rank(X) of them are positive.

*(Statement only; see e.g. Trefethen–Bau §4–5 for a constructive proof.)*

### 6.2 Definition (Moore–Penrose pseudoinverse)

Given the SVD X = U Σ Vᵀ, build Σ⁺ ∈ ℝᵖˣⁿ by inverting the *non-zero* singular values:

```
(Σ⁺)ᵢᵢ  =  1 / σᵢ      if σᵢ > 0,
(Σ⁺)ᵢᵢ  =  0           if σᵢ = 0,
(Σ⁺)ᵢⱼ  =  0           for i ≠ j.
```

The **Moore–Penrose pseudoinverse** of X is

> X⁺ := V Σ⁺ Uᵀ ∈ ℝᵖˣⁿ.   (6.1)

X⁺ exists and is unique for every X (no rank assumption required).

### 6.3 Theorem (X⁺ extends the inverse and selects the minimum-norm minimiser)

> **Theorem 6.3.**
> 
> 1. If rank(X) = p, then X⁺ = (XᵀX)⁻¹ Xᵀ. Hence θ\* = X⁺ y agrees with the OLS closed form (4.2).
> 2. For arbitrary X, the vector θ_{minnorm} := X⁺ y is the **unique** element of Θ\* with smallest Euclidean norm:
> 
>     θ_{minnorm} = argmin { ‖θ‖ : θ ∈ Θ\* }.
> 
> 3. The prediction X · X⁺ y is the orthogonal projection of y onto Col(X), regardless of rank.

**Proof sketch.**

*(1)* When rank(X) = p, all p singular values are positive and Σ⁺ Σ = Iₚ. Compute:

```
(XᵀX)⁻¹ Xᵀ  =  ( V Σᵀ Uᵀ U Σ Vᵀ )⁻¹ V Σᵀ Uᵀ
             =  V (Σᵀ Σ)⁻¹ Vᵀ V Σᵀ Uᵀ
             =  V (Σᵀ Σ)⁻¹ Σᵀ Uᵀ
             =  V Σ⁺ Uᵀ
             =  X⁺.
```

*(2)* Any θ ∈ Θ\* differs from θ_{minnorm} by an element of Null(X) (since Θ\* is the affine flat θ_{minnorm} + Null(X)). The SVD shows X⁺ y ∈ Col(Xᵀ) = Null(X)^⊥, so θ_{minnorm} ⊥ Null(X). By Pythagoras in ℝᵖ, for every v ∈ Null(X),

```
‖θ_{minnorm} + v‖²  =  ‖θ_{minnorm}‖² + ‖v‖²  ≥  ‖θ_{minnorm}‖²,
```

with equality iff v = 0.

*(3)* From the SVD, X X⁺ = U Σ Σ⁺ Uᵀ projects onto the first r columns of U, which span Col(X). ∎

### 6.4 Reading

When the design is multicollinear, the *parameters* θ are non-unique, but the *predictions* ŷ are. Col(X) is unchanged by collinearity (duplicating a column does not enlarge the span), so the projection of y onto Col(X) is unchanged; only the coordinates *inside* Col(X) become ambiguous. The pseudoinverse picks the shortest coordinate vector that lands on the projection.

This is what numerical libraries (e.g. NumPy's `lstsq` based on LAPACK's `gelsd`) compute under the hood: SVD-based, no inversion of XᵀX, no assumption of full column rank, and the minimum-norm solution as a free tiebreak.

## Takeaway

- **Model.**   ŷ = X θ, with the intercept absorbed by the bias trick X̃ = [𝟙 | X].
- **Loss.**   L(θ) = (1/n) · ‖X θ − y‖² is the negative log-likelihood under i.i.d. Gaussian noise (Theorem 2.2); OLS = MLE.
- **Gradient.**   ∇L(θ) = (2/n) · Xᵀ (X θ − y)   (Theorem 3.2).
- **Hessian.**   ∇²L(θ) = (2/n) · XᵀX — constant, PSD; PD iff rank(X) = p   (Theorem 3.3 + 3.4). Hence L is convex; strictly convex iff X has full column rank.
- **Optimum.**   Θ\* is non-empty; |Θ\*| = 1 iff rank(X) = p; in that case θ\* = (XᵀX)⁻¹ Xᵀ y   (Theorem 4.2).
- **Geometry.**   ŷ\* = H y with H := X (XᵀX)⁻¹ Xᵀ symmetric, idempotent, of rank p — the orthogonal projection onto Col(X) (Theorem 5.3). Pythagoras: ‖y‖² = ‖ŷ\*‖² + ‖r\*‖² (Corollary 5.4).
- **Singular case.**   θ_{minnorm} = X⁺ y from the SVD X = U Σ Vᵀ is the minimum-norm element of Θ\*; it agrees with (4.2) whenever rank(X) = p   (Theorem 6.3).

Next: `03_optimization.ipynb` — when the closed form is too expensive (large p) or unavailable (non-MSE loss), gradient descent walks down the convex bowl proven here. The formula ∇L(θ) = (2/n) · Xᵀ (X θ − y) derived in Theorem 3.2 is the one it uses.